In [6]:

from xml.etree import ElementTree as ET
import shutil
import os

elements_to_normalize = ["energy","cost","benifit","reliability","availability","response_time"]
# Choose the dataset to work with : 
    # Exemple :Dataset with 20 clouds and 50 services :
    #  input_dir = "./src/data/original_data/NC_20_NS_50/NC_20_NS_50_01/" 
input_dir = "./src/data/original_data/NC_5_NS_150/NC_5_NS_150_01"  

output_dir ="./src/data/preprocessedData/NC_5_NS_150/NC_5_NS_150_01"  # folder where the preprocessed data is stored

## Data preprocesssing: 

### Step 1. Copy original dataset to the output where treatment will be done  : 

In [7]:

def copy_files(source_dir, dest_dir):
    """
    Copies all files from the source directory to the destination directory,
    emptying the destination directory beforehand.

    Args:
        source_dir (str): Path to the source directory.
        dest_dir (str): Path to the destination directory.
    """

    # Check if destination directory exists, create it if it doesn't
    if not os.path.exists(dest_dir):
        os.makedirs(dest_dir)
    else:
        # Empty the destination directory
        for filename in os.listdir(dest_dir):
            file_path = os.path.join(dest_dir, filename)
            try:
                if os.path.isfile(file_path):
                    os.remove(file_path)
                elif os.path.isdir(file_path):
                    shutil.rmtree(file_path)
            except OSError as e:
                print(f"Error emptying destination directory: {e}")

    # Copy files from source directory
    for filename in os.listdir(source_dir):
        src_file_path = os.path.join(source_dir, filename)
        dest_file_path = os.path.join(dest_dir, filename)

        # Optional: Skip empty directories (if needed)
        # if not os.path.isdir(src_file_path) or len(os.listdir(src_file_path)) > 0:
        shutil.copy2(src_file_path, dest_file_path)  # Preserve file metadata (optional)

    print(f"Files copied successfully from {source_dir} to {dest_dir}")

# Example usage

copy_files(input_dir, output_dir)

print(f"Folder '{input_dir}' copied to '{output_dir}' (overwriting existing content).")


Files copied successfully from ./src/data/original_data/NC_5_NS_150/NC_5_NS_150_01 to ./src/data/preprocessedData/NC_5_NS_150/NC_5_NS_150_01
Folder './src/data/original_data/NC_5_NS_150/NC_5_NS_150_01' copied to './src/data/preprocessedData/NC_5_NS_150/NC_5_NS_150_01' (overwriting existing content).


### Step 2. Normalise data to be between 0 and 1  : 

In [8]:
import os
import xml.etree.ElementTree as ET
from collections import defaultdict

def compute_max_min(folder_path):
    # Data structure to hold service data
    services_data = defaultdict(lambda: defaultdict(list))

    # Function to parse each XML file
    def parse_xml(file_path):
        tree = ET.parse(file_path)
        root = tree.getroot()
        for service in root.find('services').findall('service'):
            service_id = service.get('id')
            for attribute in service:
                services_data[service_id][attribute.tag].append(float(attribute.text))

    # Loop through all XML files in the folder
    for file_name in os.listdir(folder_path):
        if file_name.endswith('.xml'):
            parse_xml(os.path.join(folder_path, file_name))

    # Calculate max and min values for each attribute of each service id
    result = {}
    for service_id, attributes in services_data.items():
        result[service_id] = {}
        for attr, values in attributes.items():
            result[service_id][attr] = {
                'max': max(values),
                'min': min(values)
            }

    return result

def normalize_and_modify_files(folder_path, max_min_values):
    def normalize_value_to_min(value, max_val, min_val):
        if (max_val !=  0) : 
            return (value / max_val) * -1 + (min_val / max_val)  +1 

    def normalize_value_to_max(value, max_val):
        if (max_val !=  0) : 
            return (value / max_val)      

    def modify_xml(file_path):
        tree = ET.parse(file_path)
        root = tree.getroot()
        for service in root.find('services').findall('service'):
            service_id = service.get('id')
            for attribute in service:
                attr_name = attribute.tag
                value = float(attribute.text)
                max_val = max_min_values[service_id][attr_name]['max']
                min_val = max_min_values[service_id][attr_name]['min']
                if(attr_name in ["reliability","availability","benifit"]): # Ojectifs a maximiser
                    normalized_value = normalize_value_to_max(value, max_val)
                else : # objectifs a minimiser (temps de reponse , cout , energie)
                    normalized_value = normalize_value_to_min(value, max_val, min_val)
                attribute.text = str(normalized_value)
        tree.write(file_path)

    # Loop through all XML files in the folder and modify them
    for file_name in os.listdir(folder_path):
        if file_name.endswith('.xml'):
            modify_xml(os.path.join(folder_path, file_name))

# Example usage
max_min_values = compute_max_min(output_dir)
normalize_and_modify_files(output_dir, max_min_values)

# Print results for verification
result = compute_max_min(output_dir)
for service_id, attributes in result.items():
    print(f"Service ID: {service_id}")
    for attr, values in attributes.items():
        print(f"  {attr}: Max = {values['max']}, Min = {values['min']}")


Service ID: 16
  energy: Max = 1.0, Min = 0.5122432859399685
  cost: Max = 1.0, Min = 0.27703604806408544
  benifit: Max = 1.0, Min = 0.17886020943060277
  reliability: Max = 1.0, Min = 0.8034071239865174
  availability: Max = 1.0, Min = 0.20882024184534095
  response_time: Max = 1.0, Min = 0.3121175943888972
Service ID: 24
  energy: Max = 1.0, Min = 0.5668648258236568
  cost: Max = 1.0, Min = 0.2478744277305429
  benifit: Max = 1.0, Min = 0.01716092661897768
  reliability: Max = 1.0, Min = 0.18628095551172472
  availability: Max = 1.0, Min = 0.5922538999462076
  response_time: Max = 1.0, Min = 0.1282255213856487
Service ID: 18
  energy: Max = 1.0, Min = 0.49343130705856153
  cost: Max = 1.0, Min = 0.8185053380782917
  benifit: Max = 1.0, Min = 0.27568204507143673
  reliability: Max = 1.0, Min = 0.5080680883376574
  availability: Max = 1.0, Min = 0.4739513205592957
  response_time: Max = 1.0, Min = 0.46097814776274715
Service ID: 26
  energy: Max = 1.0, Min = 0.5916482992792041
  cost:

### Back to original data :
- You may need this in practice, once you find the optimal solutions you need to display them with their original QoS not the normalized onces

In [4]:
import os
import xml.etree.ElementTree as ET
from collections import defaultdict

def compute_max_min(folder_path):
    # Data structure to hold service data
    services_data = defaultdict(lambda: defaultdict(list))

    # Function to parse each XML file
    def parse_xml(file_path):
        tree = ET.parse(file_path)
        root = tree.getroot()
        for service in root.find('services').findall('service'):
            service_id = service.get('id')
            for attribute in service:
                services_data[service_id][attribute.tag].append(float(attribute.text))

    # Loop through all XML files in the folder
    for file_name in os.listdir(folder_path):
        if file_name.endswith('.xml'):
            parse_xml(os.path.join(folder_path, file_name))

    # Calculate max and min values for each attribute of each service id
    result = {}
    for service_id, attributes in services_data.items():
        result[service_id] = {}
        for attr, values in attributes.items():
            result[service_id][attr] = {
                'max': max(values),
                'min': min(values)
            }

    return result



In [5]:
def revert_normalized_files(folder_path, max_min_values):
    def revert_value(normalized_value, max_val, min_val):
        return max_val * (1 - normalized_value) + min_val

    def modify_xml(file_path):
        tree = ET.parse(file_path)
        root = tree.getroot()
        for service in root.find('services').findall('service'):
            service_id = service.get('id')
            for attribute in service:
                attr_name = attribute.tag
                normalized_value = float(attribute.text)
                max_val = max_min_values[service_id][attr_name]['max']
                min_val = max_min_values[service_id][attr_name]['min']
                original_value = revert_value(normalized_value, max_val, min_val)
                attribute.text = str(original_value)
        tree.write(file_path)

    # Loop through all XML files in the folder and modify them
    for file_name in os.listdir(folder_path):
        if file_name.endswith('.xml'):
            modify_xml(os.path.join(folder_path, file_name))

# Example usage
max_min_values = compute_max_min(input_dir)
revert_normalized_files(output_dir, max_min_values)

# Print results for verification
result = compute_max_min(output_dir)
for service_id, attributes in result.items():
    print(f"Service ID: {service_id}")
    for attr, values in attributes.items():
        print(f"  {attr}: Max = {values['max']}, Min = {values['min']}")


Service ID: 16
  energy: Max = 17.723999999999997, Min = 9.079
  cost: Max = 2.996, Min = 0.83
  benifit: Max = 0.8031300000000001, Min = 0.14364800000000003
  reliability: Max = 10.977, Min = 8.819
  availability: Max = 9.841, Min = 2.055
  response_time: Max = 13.402, Min = 4.183
Service ID: 24
  energy: Max = 17.999, Min = 10.203
  cost: Max = 3.058, Min = 0.758
  benifit: Max = 0.5066160000000001, Min = 0.008693999999999999
  reliability: Max = 13.689, Min = 2.55
  availability: Max = 9.295, Min = 5.505
  response_time: Max = 22.632, Min = 2.902
Service ID: 18
  energy: Max = 13.473, Min = 6.648
  cost: Max = 1.124, Min = 0.92
  benifit: Max = 0.244412, Min = 0.06738
  reliability: Max = 15.803, Min = 8.029
  availability: Max = 9.655, Min = 4.576
  response_time: Max = 9.61, Min = 4.43
Service ID: 26
  energy: Max = 16.787, Min = 9.932
  cost: Max = 3.0630000000000006, Min = 2.057
  benifit: Max = 0.359456, Min = 0.045136
  reliability: Max = 20.808, Min = 11.699
  availability: M